# 🐟 Fish Audio S2 Pro - Colab 一键极速部署

本笔记本经过深度优化，专为 Colab T4 GPU 环境设计。

### 🚀 快速开始
1. **启用 GPU**: `Runtime` -> `Change runtime type` -> `T4 GPU`。
2. **设置 Token (可选)**: 点击左侧 🔑 图标 (Secrets)，添加一个名为 `HF_TOKEN` 的变量，填入你的 Hugging Face Token 并开启 "Notebook access"。这能显著加快模型下载速度。
3. **一键运行**: `Runtime` -> `Run all`。

In [ ]:
# @title 1. 核心模型环境配置 (Core Setup)
import os
from google.colab import userdata

REPO_URL = "https://github.com/entishl/fish-s2-pro-colab.git"
REPO_DIR = "fish-s2-pro-colab"

# 尝试获取 HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("✅ 成功从 Secrets 读取 HF_TOKEN")
except Exception:
    print("ℹ️ 未在 Secrets 中发现 HF_TOKEN，将使用匿名下载 (速度较慢)")

if not os.path.exists(REPO_DIR):
    print(f"正在初始化仓库: {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("正在更新代码...")
    %cd {REPO_DIR}
    !git pull
    %cd ..

%cd {REPO_DIR}

print("正在安装系统依赖...")
!apt-get update && apt-get install -y ffmpeg libsox-dev portaudio19-dev

print("正在安装 uv 包管理器...")
!pip install uv

print("正在隔离建构核心模型环境 (约需 1 分钟)...")
%cd fish-speech
# 1. 创建虚拟环境
!uv venv
# 2. 安装底层计算支撑 (cu128)
!uv sync --python 3.12 --extra cu128


In [ ]:
# @title 2. WebUI 依赖环境 (WebUI Environment)
import os
if os.path.basename(os.getcwd()) != "fish-s2-pro-colab":
    %cd fish-s2-pro-colab

print("正在安装 Web 服务和拓展依赖...")
%cd fish-speech
!uv pip install -r ../requirements.txt
%cd ..

In [ ]:
# @title 3. 启动应用 (Run Application)
import os
import sys

if os.path.basename(os.getcwd()) != "fish-s2-pro-colab":
    %cd fish-s2-pro-colab

print("正在隔离环境中启动应用...")
%cd fish-speech
!uv run python ../app.py
# 如果上面的 uv run 报错找不到依赖，也可以退回老方法：
# !./.venv/bin/python ../app.py

### 💡 小贴士
- **分步排错**: 如果卡住报错，上面的 3 个步骤可以独立重新运行。
- **模型加载**: 第一次运行会从 Hugging Face 下载约 9GB 的权重。
- **公共链接**: 运行成功后，点击最底部输出的 `public URL: https://xxxxxxxx.gradio.live` 即可开始使用。